In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()

In [2]:
origen =True
if origen:
    notas_creditos = rc.notas_creditos()
    nombre_archivo = notas_creditos['nombre_archivo']
    df = notas_creditos['archivo_salida']
else:
    # Ocultar la ventana principal de tkinter
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    # Paso 1: Seleccionar el archivo de ventas
    nombre_archivo_ventas = filedialog.askopenfilename(
        title="Por favor, ingresa el nombre del archivo de ventas (incluye la extensión .xlsx): ",
        filetypes=[("Archivos de Excel", "*.xlsx")]
    )

    # Verificar si se seleccionó un archivo
    if not nombre_archivo_ventas:
        print("No se seleccionó el archivo.")
        exit()

    # Paso 2: Cargar el archivo de ventas
    try:
        df = pd.read_excel(nombre_archivo_ventas)
        nombre_archivo = os.path.basename(nombre_archivo_ventas)
        print(f"Archivo  '{nombre_archivo}' cargado correctamente.")
    except Exception as e:
        print(f"Error al cargar el archivo: {e}")
        exit()

# Extraer el código de país y reemplazar NaN con "Desconocido" en un solo paso
df['pais'] = df['Líneas de factura/Asociado/Estado'].str.extract(r'\(([A-Z]{2})\)').fillna("Desconocido")

# Crear la columna 'total' con la lógica especificada
df['Líneas de factura/Asociado/Ciudad'] = df['Líneas de factura/Asociado/Ciudad'].astype(str).fillna("Desconocido")


# # Paso 4: Eliminar registros que no corresponden a ventas de la poción
# productos_a_eliminar = [
#     '[FLETEGRAV19] FLETE GRAVADO IVA 19% (en ventas)',
#     '[ALOJAMIENTO] SERV. ALOJAMIENTO',
#     '[reintegro] Reintegro de costos y gastos',
#     '[BKRAFT4] Bolsa de Papel Kraft Boutique 4',
#     '[PLEGP] CAJAS PLEGADIZAS POCION',
#     '[SCHT05] SACHET MSC ANCESTRAL 15 ML',
#     '[MAE20] Laminado Sachet Shampoo La Pocion 60ml',
#     '[SERVICIO ALOJAMIENTO EXTERIOR] SERVICIO ALOJAMIENTO EXTERIOR',
#     '[ACTIVOF] VENTA ACTIVO FIJO',
#     '[EXP01] FLETE INTERNACIONAL EXP',
#     '[EXP02] SEGURO INTERNACIONAL EXP',
#     '[MPE02] ENVASE PET MILK X 440 ML',
#     '[FLETE NG] FLETE NG',
#     '[EXP02] SEGURO INTERNACIONAL EXP',
#     '[EXP01] FLETE INTERNACIONAL EXP',
#     '[EXP01] Flete Internacional EXP',
#     '[EXP02] Seguro Internacional EXP',
#     '[MPE02] ENVASE PET MILK X 440 ML',
#     '[ARRENDAMIENTO INMUEBLE GRAVADO 19%] ARRENDAMIENTO INMUEBLE GRAVADO 19%' ## preguntar si se elimina
    
# ]


# df_filtrado = df[~df['Líneas de factura/Producto'].isin(productos_a_eliminar)]


# Esta linea mantiene solo los pruductos comerciales
df_filtrado = df[df['Líneas de factura/Producto'].str.startswith(('[PCN','[KD','[TNG','[B8'))]   ###### linea modificada
# Paso 6: Mostrar resumen

# '''# Paso 1: Agrupar por 'Líneas de factura/Número' y propagar el valor de 'Equipo de Ventas' hacia abajo
# df_filtrado['Equipo de Ventas'] = df_filtrado.groupby('Líneas de factura/Número')['Equipo de Ventas'].transform(lambda x: x.ffill())
# # Crear una copia explícita del DataFrame
# df_filtrado = df_filtrado.copy()

        # Guarda en la variables las ventas sin tipo de cliente y con etiqueta mayorista
etiqueta_mayorista = df_filtrado[(df_filtrado['Tipo de cliente'].isna())&
            (df_filtrado['Etiqueta contacto']=='MAYORISTA NV')
            ] 
# Copia de la etiqueta los clientes mayoristas que aparecen en blanco
df_filtrado.loc[(df_filtrado['Tipo de cliente'].isna())&
            (df_filtrado['Etiqueta contacto']=='MAYORISTA NV'), 'Tipo de cliente'
            ] = 'MAYORISTA NV'

# # Paso 2: Verificar el resultado
# print(df_filtrado[['Líneas de factura/Número', 'Equipo de Ventas']].head(20))  # Mostrar las primeras 20 filas para verificar'''
equipo_por_factura = df_filtrado.groupby('Líneas de factura/Número')['Equipo de Ventas'].first().to_dict()

# Ahora, rellenamos los valores en la columna EQUIPO_VENTAS
df_filtrado.loc[:,'Equipo de Ventas'] = df['Líneas de factura/Número'].map(equipo_por_factura)

asesora_por_factura = df_filtrado.groupby('Líneas de factura/Número')['Asesor Comercial'].first().to_dict()

# Ahora, rellenamos los valores en la columna EQUIPO_VENTAS
# df_filtrado['Asesor Comercial'] = df['Líneas de factura/Número'].map(asesora_por_factura)
df_filtrado.loc[:, 'Asesor Comercial'] = df['Líneas de factura/Número'].map(asesora_por_factura)

asesora_por_factura = df_filtrado.groupby('Líneas de factura/Número')['Tipo de cliente'].first().to_dict()

# Ahora, rellenamos los valores en la columna EQUIPO_VENTAS
# df_filtrado['Asesor Comercial'] = df['Líneas de factura/Número'].map(asesora_por_factura)
df_filtrado.loc[:, 'Tipo de cliente'] = df['Líneas de factura/Número'].map(asesora_por_factura)




# Ahora puedes modificar df_filtrado sin preocuparte por el warning
df_filtrado.loc[:, 'Líneas de factura/Fecha de factura'] = pd.to_datetime(df_filtrado['Líneas de factura/Fecha de factura'])
# df_filtrado['Líneas de factura/Fecha de factura'] = pd.to_datetime(df_filtrado['Líneas de factura/Fecha de factura'])

df_filtrado = df_filtrado.reset_index(drop=True)
# Convertir la columna 'Líneas de factura/Total' a tipo numérico
df_filtrado['Líneas de factura/Total'] = pd.to_numeric(df_filtrado['Líneas de factura/Total'], errors='coerce')
# df_filtrado.loc['', 'Líneas de factura/Total'] = pd.to_numeric(df_filtrado['Líneas de factura/Total'], errors='coerce')
# Verificar si hay valores nulos después de la conversión
print("Valores nulos en 'Líneas de factura/Total':", df_filtrado['Líneas de factura/Total'].isnull().sum())
# Paso 1: Verificar valores nulos en la columna de fecha
print("Valores nulos en 'Líneas de factura/Fecha de factura':", df_filtrado['Líneas de factura/Fecha de factura'].isnull().sum())
df_filtrado = df_filtrado.dropna(subset=['Líneas de factura/Fecha de factura'])


#  Leer el CSV desde la URL
url = "https://www.datos.gov.co/resource/32sa-8pi3.csv"
df_TRM = pd.read_csv(url)

#  Cambiar tipos de datos
df_TRM['valor'] = pd.to_numeric(df_TRM['valor'], errors='coerce')
df_TRM['unidad'] = df_TRM['unidad'].astype(str)
df_TRM['vigenciadesde'] = pd.to_datetime(df_TRM['vigenciadesde'], errors='coerce')
df_TRM['vigenciahasta'] = pd.to_datetime(df_TRM['vigenciahasta'], errors='coerce')

#  Crear una nueva columna con el año de 'vigenciadesde'
df_TRM['Año'] = df_TRM['vigenciadesde'].dt.year

#  Filtrar por el año 2025
today = pd.to_datetime('now')
df_TRM = df_TRM[df_TRM['Año'] == today.year]
df_TRM['TRM'] = df_TRM['valor']

# df_TRM = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\TRM.xlsx")

# Crear una lista para almacenar las filas expandidas
expanded_rows = []

# Iterar sobre cada fila de df_TRM y generar las fechas dentro del rango de vigencia
for _, row in df_TRM.iterrows():
    date_range = pd.date_range(start=row['vigenciadesde'], end=row['vigenciahasta'], freq='D')
    for date in date_range:
        expanded_rows.append({'Fecha': date, 'TRM': row['TRM']})

# Crear un nuevo DataFrame a partir de la lista
df_TRM_expandido = pd.DataFrame(expanded_rows)

# Eliminar duplicados (si los hay)
df_TRM_expandido = df_TRM_expandido.drop_duplicates(subset=['Fecha'])

# Ordenar por fecha
df_TRM_expandido = df_TRM_expandido.sort_values('Fecha')

# Verificar el nuevo DataFrame


df_filtrado['Líneas de factura/Fecha de factura'] = pd.to_datetime(df_filtrado['Líneas de factura/Fecha de factura'])
df_TRM_expandido['Fecha'] = pd.to_datetime(df_TRM_expandido['Fecha'])
df_filtrado = df_filtrado.sort_values(by='Líneas de factura/Fecha de factura')
df_TRM_expandido = df_TRM_expandido.sort_values(by='Fecha')

# Intentar el merge_asof
try:
    df_resultado = pd.merge_asof(
        df_filtrado,
        df_TRM_expandido[['Fecha', 'TRM']],  # Mantener solo las columnas necesarias
        left_on='Líneas de factura/Fecha de factura',
        right_on='Fecha',
        direction='backward'  # Tomar la TRM vigente más reciente anterior o igual a la fecha
    )
    
    # Verificar si la columna TRM está vacía
    if df_resultado['TRM'].isnull().all():
        print("Advertencia: La columna TRM está vacía después del merge. Verifica las fechas y los datos.")
    else:
        print("Merge completado correctamente.")
except Exception as e:
    print(f"Error al realizar el merge: {str(e)}")
    import traceback
    traceback.print_exc()
    


# # Limpiar y convertir la columna 'TRM'
# df_resultado['TRM'] = (
#     df_resultado['TRM']
#     .str.replace('.', '', regex=False)  # Eliminar puntos (separadores de miles)
#     .str.replace(',', '.', regex=False)  # Reemplazar comas por puntos (separadores decimales)
#     .astype(float)  # Convertir a tipo numérico
# )

# Verificar los valores únicos después de la conversión

# Crear la columna `total`
df_resultado['total'] = df_resultado.apply(
    lambda row: row['Líneas de factura/Total'] if row['pais'] in ['CO', 'Desconocido'] else row['Líneas de factura/Total'] * row['TRM'],
    axis=1
)


ciudad_url = "https://www.datos.gov.co/resource/gdxc-w37w.csv?$limit=5000"
DF_CIUDADES = pd.read_csv(ciudad_url)
# DF_CIUDADES = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\CIUDAD.xlsx") # Dataset con nombres correctos
DF_CIUDADES = DF_CIUDADES.rename(columns= {'nom_mpio':'Ciudad_Correcta'})
df_resultado = df_resultado.rename(columns= {'Líneas de factura/Asociado/Ciudad':'Ciudad'})

# 2️⃣ Definir los nombres de columnas
col_ciudad_correcta = "Ciudad_Correcta"  # Nombre en DF_CIUDADES
col_ciudad_ventas = "Ciudad"  # Nombre en DF_VENTAS

# 3️⃣ Convertir todas las ciudades a string y manejar NaN
df_resultado[col_ciudad_ventas] = df_resultado[col_ciudad_ventas].astype(str).fillna("Desconocido")


# 4️⃣ Lista de ciudades correctas (convertidas a string)
lista_ciudades_correctas = DF_CIUDADES[col_ciudad_correcta].astype(str).unique()

# 5️⃣ Función para encontrar la mejor coincidencia
def corregir_ciudad(ciudad_mal):
    if ciudad_mal.lower() == "nan" or ciudad_mal.strip() == "":
        return "Desconocido"  # Manejar valores vacíos o NaN
    mejor_match, score = process.extractOne(ciudad_mal, lista_ciudades_correctas)
    return mejor_match if score >= 60 else ciudad_mal  # Si el match es bajo, dejar el original

# 5️⃣ Aplicar la función a la columna de ciudades en ventas
df_resultado["Ciudad_Corregida"] = df_resultado[col_ciudad_ventas].apply(corregir_ciudad)



# 6️⃣ Convertir la columna "Ciudad_Corregida" a mayúsculas
df_resultado["Ciudad_Corregida"] = df_resultado["Ciudad_Corregida"].str.upper()

# Diccionario para renombrar las columnas
nuevos_nombres = {
    'Líneas de factura/Fecha de factura': 'Fecha_Factura',
    'Líneas de factura/Asociado': 'Cliente',
    'Líneas de factura/Número': 'Numero_Factura',
    'Líneas de factura/Producto': 'Producto',
    'Líneas de factura/Cantidad': 'Cantidad',
    'Líneas de factura/Total': 'Total',
    'Líneas de factura/Moneda/Tasa actual': 'Tasa_Cambio',
    'Líneas de factura/Asociado/Número de Identificación': 'Identificacion_Cliente',
    'Líneas de factura/Asociado/Teléfono': 'Telefono',
    'Líneas de factura/Asociado/Correo electrónico': 'Email',
    'Ciudad': 'Ciudad',
    'Líneas de factura/Asociado/Estado': 'Departamento',
    'Equipo de Ventas': 'Equipo_Ventas',
    'Líneas de factura/Referencia': 'Referencia',
    'pais': 'Pais',
    'Fecha': 'Fecha_TRM',
    'TRM': 'TRM',
    'total': 'Total($)',
    'Ciudad_Corregida': 'Ciudad_Corregida'
}

# Renombrar las columnas
df_resultado = df_resultado.rename(columns=nuevos_nombres)

# Verificar el resultado

# Extraer el día, mes y año en nuevas columnas
df_resultado['Dia'] = df_resultado['Fecha_Factura'].dt.day
df_resultado['Mes'] = df_resultado['Fecha_Factura'].dt.month
df_resultado['Año'] = df_resultado['Fecha_Factura'].dt.year


# Reorganizar las columnas si es necesario
column_order = ['Numero_Factura','Fecha_Factura', 'Dia', 'Mes', 'Año', 'Cliente','Identificacion_Cliente','Producto', 'Cantidad', 
                'Total', 'Tasa_Cambio','TRM', 'Total($)','Telefono', 'Email','Pais','Ciudad', 'Ciudad_Corregida', 'Departamento', 
                'Equipo_Ventas', 'Referencia', 'Asesor Comercial', 'Tipo de cliente'] ## aqui se agregregan las nuevas columnas
df_resultado = df_resultado[column_order]



# Convertir la columna "Cliente" a mayúsculas
df_resultado['Cliente'] = df_resultado['Cliente'].str.upper()

# Eliminar espacios en blanco al principio y al final de cada valor en la columna "Cliente"
df_resultado['Cliente'] = df_resultado['Cliente'].str.strip()
# Convertir la columna "Producto" a mayúsculas
df_resultado['Producto'] = df_resultado['Producto'].str.upper()
# Eliminar espacios en blanco al principio y al final de cada valor en la columna "Producto"
df_resultado['Producto'] = df_resultado['Producto'].str.strip()
# Convertir los nombres de las columnas a mayúsculas
df_resultado.columns = df_resultado.columns.str.upper()



Archivo de ventas 'agosto.xlsx' cargado correctamente.
Archivo de notas crédito 'notas_agosto.xlsx' cargado correctamente.
Archivo consolidado guardado como 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\RAW DATA\PROCESADO\agosto_procesado.xlsx'.
Facturas afectadas por notas crédito:
(350, 4)
Listado de facturas afectadas guardado como 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\RAW DATA\FACTURAS AFECTADAS\agosto_facturas_afectadas.xlsx'.
Valores nulos en 'Líneas de factura/Total': 0
Valores nulos en 'Líneas de factura/Fecha de factura': 0
Merge completado correctamente.


In [3]:

# Limpieza en tu DataFrame actual
df_resultado['IDENTIFICACION_CLIENTE'] = (
    df_resultado['IDENTIFICACION_CLIENTE']
    .astype(str)  # Convertir a string
    .str.strip()  # Eliminar espacios al principio y al final
    .str.replace(r'\s+', '', regex=True)  # Eliminar espacios adicionales entre caracteres
)

df_resultado['IDENTIFICACION_CLIENTE'] = df_resultado['IDENTIFICACION_CLIENTE'].apply(rc.limpiar_documento)



In [4]:

# Primero, obtenemos la lista de columnas
columnas = df_resultado.columns.tolist()

# Encontramos la posición de la columna "Producto"
posicion_producto = columnas.index('PRODUCTO')

# Movemos "Categoría" antes de "Producto"
columnas.insert(posicion_producto, columnas.pop(columnas.index('TIPO DE CLIENTE')))

# Reorganizamos el DataFrame
df_resultado = df_resultado[columnas]

# Rellenar los valores NaN en "Categoría" cuando EQUIPO_VENTAS sea "Shopify"
df_resultado.loc[(df_resultado['TIPO DE CLIENTE'].isna()) & (df_resultado['EQUIPO_VENTAS'] == 'Shopify'), 'TIPO DE CLIENTE'] = 'SHOPIFY'
# Rellenar los valores NaN en "Categoría" cuando EQUIPO_VENTAS sea "Shopify"
df_resultado.loc[df_resultado['EQUIPO_VENTAS'] == 'Punto de venta', 'TIPO DE CLIENTE'] = 'CALL CENTER'


# df_resultado[df_resultado['CATEGORÍA'].isna()].to_excel(r"C:\Users\Dataa\Desktop\ventas_sin_categoria.xlsx")

df_resultado.loc[~df_resultado['PAIS'].isin(['CO', 'Desconocido']), 'TIPO DE CLIENTE'] = df_resultado['PAIS']


# Mostrar las primeras filas para verificar los cambios


In [ ]:
# Esta Linea de codigo 
# Agrgrer varificacion de vendedores mayoristas como asesor comercial
asesores_moyorista = df_resultado[df_resultado['TIPO DE CLIENTE'] == 'MAYORISTA NV']['ASESOR COMERCIAL'].drop_duplicates().tolist()
asesores_moyorista = [a for a in asesores_moyorista if a is not None]
asesores_sin_categoria = df_resultado[(df_resultado['TIPO DE CLIENTE'].isna())&(df_resultado['ASESOR COMERCIAL'].isin(asesores_moyorista))]
df_resultado.loc[(df_resultado['TIPO DE CLIENTE'].isna())&(df_resultado['ASESOR COMERCIAL'].isin(asesores_moyorista)), 'TIPO DE CLIENTE'] = 'MAYORISTA NV'

In [31]:

# Rellenar los valores vacíos en "Categoría" con "Call center"
df_resultado['TIPO DE CLIENTE'].fillna('CALL CENTER', inplace=True)   ### REVISAR


# 9. Eliminar las columnas "REFERENCIA" y "Número de Identificación"
# df_resultado.drop(columns=['Número de Identificación'], inplace=True)
# print(f"- Registros originales: {len(df_resultado)}")

# df_final = df_resultado[df_resultado["PRODUCTO"] != "[MPE02] ENVASE PET MILK X 440 ML"]

# # Guardar el DataFrame en un archivo Excel
# df_final.to_excel(f"VENTAS_{nombre_archivo}", index=False)

df_resultado= df_resultado.rename(columns={'TIPO DE CLIENTE':'CATEGORÍA'})


C:\Users\Dataa\AppData\Local\Temp\ipykernel_26236\3103707366.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_resultado['TIPO DE CLIENTE'].fillna('CALL CENTER', inplace=True)   ### REVISAR


In [33]:

# validar clientes nuevos
try:
    ruta = rc.validar_ruta()
    ruta_historoica = ruta / 'file' / f'ventas_historicas.xlsx'

    

    ventas_historicas_agru=pd.read_excel(ruta_historoica)
    ventas_historicas_agru['FECHA_FACTURA'] = pd.to_datetime(ventas_historicas_agru['FECHA_FACTURA'])

    ventas_historicas_agru['IDENTIFICACION_CLIENTE'] =  ventas_historicas_agru['IDENTIFICACION_CLIENTE'].apply(rc.limpiar_documento)
    ventas_historicas_agru['IDENTIFICACION_CLIENTE'].astype(str)



    df_resultado = df_resultado.merge(ventas_historicas_agru, on='IDENTIFICACION_CLIENTE', how='left', suffixes=('', '_FECHA_MIN'))

    now = pd.to_datetime('now')

    df_resultado['CLIENTES NUEVOS'] = np.where(
        ((df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.month == now.month) & 
        (df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.year == now.year))|(df_resultado['FECHA_FACTURA_FECHA_MIN'].isna()),
        "Cliente sin historico",
        "No es cliente nuevo"
    )
except Exception as e:
    print(f"Error al validar clientes nuevos: {e}")

# return  {'Base':df_resultado,
#         'nombre_archivo':notas_creditos['nombre_archivo'],
#         'facturas_afectadas':notas_creditos['facturas_afectadas'],
#         'errores':etiqueta_mayorista
#             }



In [ ]:
ruta = rc.validar_ruta()
ruta_historoica = ruta / 'file' / f'ventas_historicas.xlsx'



ventas_historicas_agru=pd.read_excel(ruta_historoica)
ventas_historicas_agru['FECHA_FACTURA'] = pd.to_datetime(ventas_historicas_agru['FECHA_FACTURA'])

ventas_historicas_agru['IDENTIFICACION_CLIENTE'] =  ventas_historicas_agru['IDENTIFICACION_CLIENTE'].apply(rc.limpiar_documento)
ventas_historicas_agru['IDENTIFICACION_CLIENTE'].astype(str)



df_resultado = df_resultado.merge(ventas_historicas_agru, on='IDENTIFICACION_CLIENTE', how='left', suffixes=('', '_FECHA_MIN'))

now = pd.to_datetime('now')

df_resultado['CLIENTES NUEVOS'] = np.where(
    ((df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.month == now.month) & 
    (df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.year == now.year))|(df_resultado['FECHA_FACTURA_FECHA_MIN'].isna()),
    "Cliente sin historico",
    "No es cliente nuevo"
)

In [39]:
df_resultado['CLIENTES NUEVOS'] = np.where(
                ((df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.month == now.month) & 
                (df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.year == now.year))|((df_resultado['FECHA_FACTURA_FECHA_MIN'].isna())&(df_resultado['CATEGORÍA']=='MAYORISTA NV')),
                "Cliente Nuevo",
                ""
            )

In [40]:
df_resultado.shape

(24375, 25)

In [41]:
df_resultado['CLIENTES NUEVOS'].value_counts()

CLIENTES NUEVOS
                 22531
Cliente Nuevo     1844
Name: count, dtype: int64